# Projet : Data Preparation for US Flight Delay Prediction

## Notations du papier
| Symbole | Définition |
|---------|------------|
| **FT** | Flight Table (AOTP filtré) |
| **OT** | Weather Observation Table (QCLCD filtré) |
| **JT** | Joint Table = résultat du join FT ⋈ OT |
| **Wo** | Array météo aéroport départ : `[O(Ao, tsd), O(Ao, tsd-1h), ..., O(Ao, tsd-12h)]` — 13 snapshots |
| **Wd** | Array météo aéroport destination : `[O(Ad, tsa), O(Ad, tsa-1h), ..., O(Ad, tsa-12h)]` — 13 snapshots |
| **F** | `⟨Ao, Ad, tsd, tad, tsa, taa⟩` infos du vol |
| **C** | Label cible : on-time si `AD(F) < Th`, delayed si `AD(F) ≥ Th` |
| **k** | Clé composite du join MapReduce : `⟨(Airport, Date), table_tag⟩` |
| **Th** | Seuil de retard (15, 30, 45, 60, 90 min dans le papier) |

---
##Installation et initialisation de Spark

In [0]:


import os, subprocess
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH']      = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[*]')
         .appName('FlightDelayPrep')
         .config('spark.ui.showConsoleProgress', 'false')
         .getOrCreate())

sc = spark.sparkContext
print('Spark version :', spark.version)
print(subprocess.check_output(['java','-version'],
       stderr=subprocess.STDOUT).decode().splitlines()[0])

In [0]:
from pyspark.sql.functions import (
    col, trim, when, coalesce, lit,
    to_date, to_timestamp, concat_ws, lpad,
    unix_timestamp, count, sum as spark_sum,
    round as spark_round
)
from pyspark.sql.types import IntegerType, DoubleType, StringType


In [0]:
!wget -O dataset.zip "https://www.dropbox.com/scl/fo/7naldic5mjgf6rxybc8iu/AH-BHFhgrBic53yzJ3xR8vQ?rlkey=v3btf4zxdmg44otq1xtn0nu9q&e=1&st=chag2gmi"
!unzip dataset.zip

In [0]:
flights_raw = (spark.read
               .format('csv')
               .option('header', 'true')
               .option('inferSchema', 'true')
               .load('/Volumes/workspace/default/flights/'))

print(f'AOTP  : {flights_raw.count():,} lignes | {len(flights_raw.columns)} colonnes')
flights_raw.show(3, truncate=True)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
weather_raw = (spark.read
               .format('csv')
               .option('header', 'true')
               .option('inferSchema', 'false')  # tout en String car valeurs mixtes
               .load('/Volumes/workspace/default/weather/'))

print(f'QCLCD : {weather_raw.count():,} lignes | {len(weather_raw.columns)} colonnes')
print('Colonnes :', weather_raw.columns)
weather_raw.show(3)

In [0]:
wban_mapping = (spark.read
                .format('csv')
                .option('header', 'true')
                .option('inferSchema', 'true')
                .load('/Volumes/workspace/default/airport_time_zone/wban_airport_timezone.csv'))

print(f'WBAN mapping : {wban_mapping.count():,} aéroports')
print('Colonnes :', wban_mapping.columns)
wban_mapping.show(5)

In [0]:
print('=== Schéma flights ===')
flights_raw.printSchema()

print('\n=== Schéma weather) ===')
weather_raw.printSchema()

print('\n=== Schéma WBAN  ===')
wban_mapping.printSchema()

In [0]:
import torch
torch.cuda.is_available()

---
##  Préprocessing AOTP → FT (Section 4.1 du papier)

1. Filtrer les vols **annulés** (`CANCELLED=1`) et **détournés** (`DIVERTED=1`)
2. Supprimer ces colonnes (inutiles après filtrage)
3. Traiter les valeurs manquantes dans `WEATHER_DELAY` et `NAS_DELAY`
4. Créer le label **C** : `ARR_DELAY_NEW ≥ Th` → delayed (1), sinon on-time (0)
5. Construire les timestamps `tsd` (départ planifié) et `tsa` (arrivée planifiée)

In [0]:
# Seuil de retard Th = 15 minutes (seuil FAA, utilisé dans le papier)
Th = 15

# Filtrer annulés et détournés

FT = (flights_raw
      .filter((col('CANCELLED') == 0) & (col('DIVERTED') == 0))
      .drop('CANCELLED', 'DIVERTED'))

print(f'AOTP brut : {flights_raw.count():,} vols')
print(f'FT filtré : {FT.count():,} vols')
print(f'Supprimés : {flights_raw.count() - FT.count():,} (annulés + détournés)')

In [0]:
# Traiter les valeurs manquantes

FT = (FT
      .withColumn('WEATHER_DELAY', coalesce(col('WEATHER_DELAY').cast('double'), lit(0.0)))
      .withColumn('NAS_DELAY',     coalesce(col('NAS_DELAY').cast('double'),     lit(0.0)))
      .withColumn('ARR_DELAY_NEW', coalesce(col('ARR_DELAY_NEW').cast('double'), lit(0.0))))

FT.select('ARR_DELAY_NEW', 'WEATHER_DELAY', 'NAS_DELAY').describe().show()

In [0]:
from pyspark.sql.functions import when, col

# Créer le label C
#        C = 1 (delayed)  si ARR_DELAY_NEW >= Th
#        C = 0 (on-time)  si ARR_DELAY_NEW  < Th
FT = FT.withColumn('C', when(col('ARR_DELAY_NEW') >= Th, 1).otherwise(0))

print(f' label C (Th={Th} min) ')
label_dist = FT.groupBy('C').count().orderBy('C').collect()
total = sum(r['count'] for r in label_dist)
for r in label_dist:
    label = 'On-time' if r['C']==0 else 'Delayed'
    print(f'  C={r["C"]} ({label}) : {r["count"]:,} vols ({r["count"]/total*100:.1f}%)')

In [0]:
# Construire les timestamps tsd et tsa
#        Nécessaires pour la fenêtre météo [t, t-12h]
FT = FT.withColumn(
    'tsd',  # scheduled departure time
    to_timestamp(
        concat_ws(' ',
            col('FL_DATE').cast('string'),
            lpad(col('CRS_DEP_TIME').cast('string'), 4, '0')
        ),
        'yyyy-MM-dd HHmm'
    )
)
FT.select('FL_DATE', 'ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID',
          'CRS_DEP_TIME', 'tsd', 'ARR_DELAY_NEW', 'C').show(5)
print(f'FT : {FT.count():,} lignes | {len(FT.columns)} colonnes')

---
##  Préprocessing QCLCD → OT (Section 4.1 du papier)

### Problème de jointure
Les fichiers météo identifient chaque station par son **numéro WBAN** (ex: `'03011'`)  
mais les vols utilisent des **AirportID numériques** (ex: `12478`).  

La table `wban-airport-timezone.csv` fait le pont :  
```
weather.WBAN (str '03011') → cast int(3011) = wban_mapping.WBAN → wban_mapping.AirportID = flights.ORIGIN_AIRPORT_ID
```

Ensuite on filtre OT pour **ne garder que les aéroports présents dans FT**.

In [0]:
#  Nettoyer weather_raw et convertir les types
weather_clean = (weather_raw
    .withColumn('WBAN',           trim(col('WBAN')))
    .withColumn('WBAN_INT',       col('WBAN').try_cast('int'))      # '03011' → 3011
    .withColumn('DryBulbCelsius', col('DryBulbCelsius').try_cast('double'))  # T
    .withColumn('WindSpeed',      col('WindSpeed').try_cast('double'))       # Ws
    .withColumn('Visibility',     col('Visibility').try_cast('double'))      # V
    .withColumn('HourlyPrecip',   col('HourlyPrecip').try_cast('double'))    # P
    .withColumn('RelativeHumidity', col('RelativeHumidity').try_cast('double')) # H
    # Date QCLCD format YYYYMMDD → DateType
    .withColumn('Date', to_date(col('Date').try_cast('string'), 'yyyyMMdd'))
    # Time = HHMM string (ex: '0055', '2330')
    .withColumn('Time', lpad(trim(col('Time')), 4, '0'))
)

print(f'weather_clean : {weather_clean.count():,} observations')
weather_clean.select('WBAN', 'WBAN_INT', 'Date', 'Time','DryBulbCelsius', 'WindSpeed', 'Visibility').show(5)

In [0]:
#  Join weather × wban_mapping
#        weather.WBAN_INT (int) = wban_mapping.WBAN (int)
#        → ajoute la colonne AirportID à chaque observation météo
weather_with_airport = (weather_clean
    .join(
        wban_mapping.select(
            col('WBAN').alias('WBAN_MAP'),
            col('AirportID').alias('AirportID'),
            col('TimeZone').alias('TimeZone')
        ),
        weather_clean['WBAN_INT'] == col('WBAN_MAP'),
        'inner'
    )
    .drop('WBAN_MAP', 'WBAN_INT')
)

print(f'Observations avec AirportID : {weather_with_airport.count():,}')
print(f'(sans correspondance WBAN : {weather_clean.count() - weather_with_airport.count():,} supprimées)')
weather_with_airport.select('WBAN', 'AirportID', 'Date', 'Time',
                             'DryBulbCelsius', 'WindSpeed').show(5)

In [0]:
# Filtrer OT : garder uniquement les aéroports présents dans FT
#        FT utilise ORIGIN_AIRPORT_ID et DEST_AIRPORT_ID
#        OT utilise AirportID

airports_origin = FT.select(col('ORIGIN_AIRPORT_ID').alias('AirportID'))
airports_dest   = FT.select(col('DEST_AIRPORT_ID').alias('AirportID'))

airports_valid  = airports_origin.union(airports_dest).distinct()

print(f'Aéroports : {airports_valid.count()}')

OT = (weather_with_airport
      .join(airports_valid,
            weather_with_airport['AirportID'] == airports_valid['AirportID'],
            'inner')
      .drop(airports_valid['AirportID']))

print(f'QCLCD   : {weather_with_airport.count():,} observations')
print(f'OT filtré    : {OT.count():,} observations')
print(f'Supprimées   : {weather_with_airport.count() - OT.count():,} (hors aéroports FT)')

In [0]:
# Construire le timestamp t de chaque observation météo
#        O = ⟨A, t, T, H, Ws, P, S, V, D⟩  (Définition 2.2 du papier)

OT = OT.withColumn(
    't',  # observation time (notation papier)
    to_timestamp(
        concat_ws(' ',
            col('Date').cast('string'),
            col('Time')
        ),
        'yyyy-MM-dd HHmm'
    )
)

print(' OT final ')
print(f'Lignes : {OT.count():,} | Colonnes : {len(OT.columns)}')
OT.select('AirportID', 'WBAN', 'Date', 'Time', 't',
          'DryBulbCelsius', 'WindSpeed', 'Visibility', 'HourlyPrecip', 'SkyCondition').show(5)

---
## Étape 5 — Exploration des données (Section 3 du papier)

Analyse de la distribution des retards et des 5 catégories de causes.

In [0]:
from pyspark.sql.functions import year as spark_year

# % on-time / delayed par année ──
print(' Performance des vols par année ')
(FT
 .withColumn('Year', spark_year(col('FL_DATE').cast('date')))
 .groupBy('Year')
 .agg(
     count('*').alias('Total'),
     spark_round(count(when(col('C')==0, 1)) * 100.0 / count('*'), 1).alias('OnTime_%'),
     spark_round(count(when(col('C')==1, 1)) * 100.0 / count('*'), 1).alias('Delayed_%')
 )
 .orderBy('Year')
 .show()
)


In [0]:

# causes de retard
print('Causes de retard (sur vols retardés) ')
delayed_only = FT.filter(col('C') == 1)

totals = delayed_only.agg(
    spark_sum('WEATHER_DELAY').alias('weather'),
    spark_sum('NAS_DELAY').alias('nas')
).collect()[0]

grand = (totals['weather'] or 0) + (totals['nas'] or 0)
if grand > 0:
    print(f'  WeatherDelay : {(totals["weather"] or 0)/grand*100:.1f}%')
    print(f'  NASDelay     : {(totals["nas"]     or 0)/grand*100:.1f}%')
else:
    print(f'  WeatherDelay total : {totals["weather"]:,.0f} min')
    print(f'  NASDelay total     : {totals["nas"]:,.0f} min')

---
## Étape 6 — Join MapReduce → JT (Algorithme 1, Section 4.1 du papier)

### Principe du improved repartition join (Figure 3 du papier)

```
Phase MAP :
  Si tuple ∈ OT :  émettre  k=((AirportID, Date), "OT"),  V=("OT", tuple)
  Si tuple ∈ FT :  émettre  k=((AirportID, Date), "FT"),  V=("FT", tuple)

Phase PARTITION : hash(join_key) mod #reducers

Phase REDUCE : pour chaque (AirportID, Date)
  1. Stocker toutes les obs OT dans un array AO trié par t
  2. Pour chaque vol FT : AT = get_hourly_obs(AO, tsd) → 13 snapshots
  3. Émettre merge(f, AT)
```

---
### JOIN STEP 1 — VERSION RDD (fidèle à l'Algorithme 1 du papier)
### Clé : (ORIGIN_AIRPORT_ID, FL_DATE)  pour FT  (AirportID,Date)     pour OT

### Phase MAP : tagger chaque tuple avec son origine


In [0]:

# FT : clé = (Ao=ORIGIN_AIRPORT_ID, Date(tsd)=FL_DATE)
ft_tagged = FT.rdd.map(lambda row: (
    (int(row['ORIGIN_AIRPORT_ID']), str(row['FL_DATE'])),  # join_key = ⟨Ao, Date⟩
    ('FT', row.asDict())                                    # table_tag + contenu
))

# OT : clé = (A=AirportID, Date(t)=Date)
ot_tagged = OT.rdd.map(lambda row: (
    (int(row['AirportID']), str(row['Date'])),  # join_key = ⟨A, Date⟩
    ('OT', row.asDict())                         # table_tag + contenu
))

print(' Exemples de paires (clé composite, valeur) ')
print('\n[FT] :')
for k, v in ft_tagged.take(2):
    r = v[1]
    print(f'  k=({k[0]}, {k[1]})  tag={v[0]}  vol={r.get("OP_CARRIER_FL_NUM")}  {k[0]}→{r.get("DEST_AIRPORT_ID")}')

print('\n[OT] :')
for k, v in ot_tagged.take(2):
    r = v[1]
    print(f'  k=({k[0]}, {k[1]})  tag={v[0]}  WBAN={r.get("WBAN")}  t={r.get("Time")}')

In [0]:
# UNION des deux RDDs ──
combined_step1 = ft_tagged.union(ot_tagged)

print(f'Paires (k,V) total : {combined_step1.count():,}')
print(f'  FT : {ft_tagged.count():,}')
print(f'  OT : {ot_tagged.count():,}')

# Phase PARTITION + REDUCE : groupByKey ──
# Toutes les lignes de même (AirportID, Date) sont groupées
grouped_step1 = combined_step1.groupByKey().mapValues(list)

print(f'\nClés (AirportID, Date) distinctes : {grouped_step1.count():,}')

# Aperçu d'un groupe
k_sample, v_sample = grouped_step1.take(1)[0]
ft_recs = [r for tag, r in v_sample if tag=='FT']
ot_recs = [r for tag, r in v_sample if tag=='OT']
print(f'\nExemple — clé={k_sample} : {len(ft_recs)} vols FT, {len(ot_recs)} obs OT')

In [0]:
from datetime import datetime, timedelta

def get_hourly_observations(ot_records, tsd_val, n_obs=13):
    """
    Algorithme 1 du papier — get_hourly_observations(AO, f.tsd)

    Pour un vol partant à tsd, retourne les n_obs observations météo horaires
    les plus proches de : tsd, tsd-1h, tsd-2h, ..., tsd-12h  (13 snapshots)

    Wo = [O(Ao,tsd), O(Ao,tsd-1h), ..., O(Ao,tsd-12h)]
    """
    if tsd_val is None:
        return [None] * n_obs
    try:
        tsd = datetime.fromisoformat(str(tsd_val))
    except:
        return [None] * n_obs

    target_times = [tsd - timedelta(hours=i) for i in range(n_obs)]

    result = []
    for target in target_times:
        best, best_diff = None, float('inf')
        for _, obs in ot_records:
            t_obs = obs.get('t')
            if t_obs is None:
                continue
            try:
                obs_dt = datetime.fromisoformat(str(t_obs))
                diff   = abs((obs_dt - target).total_seconds())
                if diff < best_diff:
                    best_diff = diff
                    best = obs
            except:
                continue
        result.append(best)
    return result

In [0]:
def reduce_step1(key_records):
    """
    Fonction Reduce

    Pour chaque clé (ORIGIN_AIRPORT_ID, FL_DATE) :
    1. Stocker les observations OT dans AO trié par t
    2. Pour chaque vol FT : AT = get_hourly_obs(AO, tsd) → 13 snapshots
    3. Émettre merge(f, AT) : tuple enrichi avec Wo
    """
    key, records = key_records

    ot_recs = [(tag, r) for tag, r in records if tag == 'OT']
    ft_recs = [(tag, r) for tag, r in records if tag == 'FT']

    # Trier AO par temps d'observation (comme dans l'algo du papier)
    ot_sorted = sorted(ot_recs, key=lambda x: str(x[1].get('t', '')))

    results = []
    for _, flight in ft_recs:
        # AT = 13 snapshots météo horaires avant le décollage
        Wo = get_hourly_observations(ot_sorted, flight.get('tsd'))

        merged = dict(flight)  # copie du tuple vol
        for i, obs in enumerate(Wo):
            # Wo_0h = tsd, Wo_1h = tsd-1h, ..., Wo_12h = tsd-12h
            pfx = f'Wo_{i}h_'
            if obs:
                merged[pfx + 'Temp']     = obs.get('DryBulbCelsius')
                merged[pfx + 'Wind']     = obs.get('WindSpeed')
                merged[pfx + 'Vis']      = obs.get('Visibility')
                merged[pfx + 'Precip']   = obs.get('HourlyPrecip')
                merged[pfx + 'Sky']      = obs.get('SkyCondition')
            else:
                for suf in ['Temp', 'Wind', 'Vis', 'Precip', 'Sky']:
                    merged[pfx + suf] = None
        results.append(merged)

    return results


In [0]:
JT_step1_rdd = grouped_step1.flatMap(reduce_step1)

print(f'JT après step 1 (vol + Wo) : {JT_step1_rdd.count():,} tuples')
s = JT_step1_rdd.take(1)[0]
print(f"\nExemple — vol {s.get('ORIGIN_AIRPORT_ID')}→{s.get('DEST_AIRPORT_ID')}")
print(f"  C={s.get('C')}  tsd={s.get('tsd')}")
print(f"  Wo_0h_Temp={s.get('Wo_0h_Temp')}  Wo_0h_Wind={s.get('Wo_0h_Wind')}")
print(f"  Wo_6h_Temp={s.get('Wo_6h_Temp')}  Wo_12h_Temp={s.get('Wo_12h_Temp')}")

### 5.2 — Join Step 2 : JT_step1 ⋈ OT sur clé ⟨Ad (Dest), Date(tsa)⟩ → météo destination Wd

In [0]:
# ══════════════════════════════════════════════════════════════
# JOIN STEP 2 — Ajouter Wd (météo destination)
# Clé : (DEST_AIRPORT_ID, FL_DATE)  pour JT_step1
#       (AirportID, Date)            pour OT
# ══════════════════════════════════════════════════════════════

# JT_step1 taggué avec clé = (Dest, Date)
jt1_tagged = JT_step1_rdd.map(lambda row: (
    (int(row.get('DEST_AIRPORT_ID', 0)), str(row.get('FL_DATE', ''))),
    ('FT', row)
))

# OT taggué (même que step 1)
ot_tagged_s2 = OT.rdd.map(lambda row: (
    (int(row['AirportID']), str(row['Date'])),
    ('OT', row.asDict())
))

combined_step2 = jt1_tagged.union(ot_tagged_s2)
grouped_step2  = combined_step2.groupByKey().mapValues(list)


def reduce_step2(key_records):
    """Reduce step 2 : ajoute Wd (météo destination) à chaque tuple."""
    key, records = key_records

    ot_recs = [(tag, r) for tag, r in records if tag == 'OT']
    ft_recs = [(tag, r) for tag, r in records if tag == 'FT']
    ot_sorted = sorted(ot_recs, key=lambda x: str(x[1].get('t', '')))

    results = []
    for _, flight in ft_recs:
        # tsa = tsd + CRS_ELAPSED_TIME (en minutes)
        tsd_val  = flight.get('tsd')
        elapsed  = flight.get('CRS_ELAPSED_TIME')
        if tsd_val and elapsed:
            try:
                tsa_val = datetime.fromisoformat(str(tsd_val)) + timedelta(minutes=float(elapsed))
            except:
                tsa_val = None
        else:
            tsa_val = None

        Wd = get_hourly_observations(ot_sorted, tsa_val)

        merged = dict(flight)
        for i, obs in enumerate(Wd):
            pfx = f'Wd_{i}h_'
            if obs:
                merged[pfx + 'Temp']   = obs.get('DryBulbCelsius')
                merged[pfx + 'Wind']   = obs.get('WindSpeed')
                merged[pfx + 'Vis']    = obs.get('Visibility')
                merged[pfx + 'Precip'] = obs.get('HourlyPrecip')
                merged[pfx + 'Sky']    = obs.get('SkyCondition')
            else:
                for suf in ['Temp', 'Wind', 'Vis', 'Precip', 'Sky']:
                    merged[pfx + suf] = None
        results.append(merged)
    return results


JT_rdd = grouped_step2.flatMap(reduce_step2)

print(f'JT final (F + Wo + Wd + C) : {JT_rdd.count():,} tuples')
s = JT_rdd.take(1)[0]
print(f"\nExemple tuple JT complet :")
print(f"  {s.get('ORIGIN_AIRPORT_ID')}→{s.get('DEST_AIRPORT_ID')}  C={s.get('C')}")
print(f"  Wo_0h : Temp={s.get('Wo_0h_Temp')} Wind={s.get('Wo_0h_Wind')} Vis={s.get('Wo_0h_Vis')}")
print(f"  Wd_0h : Temp={s.get('Wd_0h_Temp')} Wind={s.get('Wd_0h_Wind')} Vis={s.get('Wd_0h_Vis')}")

---
## Étape 6 — Version DataFrame optimisée du Join

La version RDD ci-dessus est **fidèle à l'Algorithme 1** du papier.
On montre maintenant la version **DataFrame** plus performante (moins de shuffles, comme les requêtes optimisées q7/q8 du TP DataFrames).

In [0]:
# ── Join Step 1 DataFrame : FT ⋈ OT sur (ORIGIN_AIRPORT_ID=AirportID, FL_DATE=Date) ──
# Condition temporelle : obs météo dans [tsd-12h, tsd]

JT_df = (FT.alias('f')
    .join(
        OT.alias('wo'),
        on=(
            (col('f.ORIGIN_AIRPORT_ID') == col('wo.AirportID')) &
            (col('f.FL_DATE').cast('date') == col('wo.Date')) &
            # Fenêtre Wo : obs entre tsd-12h et tsd
            (
                unix_timestamp(col('f.tsd')) - unix_timestamp(col('wo.t'))
            ).between(0, 12 * 3600)
        ),
        how='left'
    )
    .select(
        col('f.*'),
        col('wo.t').alias('Wo_t'),
        col('wo.DryBulbCelsius').alias('Wo_Temp'),
        col('wo.WindSpeed').alias('Wo_Wind'),
        col('wo.Visibility').alias('Wo_Vis'),
        col('wo.HourlyPrecip').alias('Wo_Precip'),
        col('wo.SkyCondition').alias('Wo_Sky')
    )
)

print(f'Après Join Step 1 (DataFrame) : {JT_df.count():,} lignes')
JT_df.select('ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID', 'tsd', 'Wo_t',
             'Wo_Temp', 'Wo_Wind', 'Wo_Vis', 'C').show(5)

In [0]:
# ── Join Step 2 DataFrame : JT ⋈ OT sur (DEST_AIRPORT_ID=AirportID, FL_DATE=Date) ──
# + construction de tsa = tsd + CRS_ELAPSED_TIME

from pyspark.sql.functions import expr

JT_df = JT_df.withColumn(
    'tsa',  # estimated scheduled arrival time
    (unix_timestamp(col('tsd')) + col('CRS_ELAPSED_TIME').cast('double') * 60)
    .cast('timestamp')
)

JT_final_df = (JT_df.alias('jt')
    .join(
        OT.alias('wd'),
        on=(
            (col('jt.DEST_AIRPORT_ID') == col('wd.AirportID')) &
            (col('jt.FL_DATE').cast('date') == col('wd.Date')) &
            # Fenêtre Wd : obs entre tsa-12h et tsa
            (
                unix_timestamp(col('jt.tsa')) - unix_timestamp(col('wd.t'))
            ).between(0, 12 * 3600)
        ),
        how='left'
    )
    .select(
        col('jt.*'),
        col('wd.t').alias('Wd_t'),
        col('wd.DryBulbCelsius').alias('Wd_Temp'),
        col('wd.WindSpeed').alias('Wd_Wind'),
        col('wd.Visibility').alias('Wd_Vis'),
        col('wd.HourlyPrecip').alias('Wd_Precip'),
        col('wd.SkyCondition').alias('Wd_Sky')
    )
)

print(f'JT final (DataFrame) : {JT_final_df.count():,} lignes')
JT_final_df.select(
    'ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID',
    'Wo_Temp', 'Wo_Wind', 'Wd_Temp', 'Wd_Wind', 'C'
).show(5)